In [ ]:
import os
import polars as pl
from anngeno import AnnGeno

## Set new annotations for AnnGeno

In [ ]:
anngeno_path = "/home/dnanexus/data_dir/anngeno_training.ag"
ag = AnnGeno(anngeno_path,  filemode="r", low_mem=True)
ag

In [ ]:
r = ag.get_region('ENSG00000123739')
r['annotations']

In [ ]:
import matplotlib.pyplot as plt

# Get non-null elements
absplice_dna_max = r['annotations']['AbSplice2_max'].drop_nulls()

# Plot histogram
plt.hist(absplice_dna_max.to_numpy(), bins=30, log=True)

## Get the small set of associations

In [ ]:
abs2_genes = abs2_annos.select(pl.col('region').unique().sort()).collect()
abs2_genes

In [ ]:
plist = []
for pheno in os.listdir('/home/dnanexus/data_dir/phenotypes_corr'):
    if pheno.endswith('.parquet'):
        plist.append(pheno[:-22])
plist

In [ ]:
assocs = pl.read_parquet('/home/dnanexus/data_dir/genebass_continuous_associations_ukbbgym.pq')
assocs = assocs.with_columns(
    pl.col("description").str.to_lowercase().str.replace_all(" ", "_").alias("phenotype")
).filter(
    pl.col('annotation').str.contains("pLoF")
)
assocs

In [ ]:
abs2_assocs = assocs.filter(pl.col('gene_id').is_in(abs2_genes['region']) & pl.col('phenotype').is_in(plist))
abs2_assocs

In [ ]:
abs2_assocs.write_parquet('/home/dnanexus/data_dir/absplice2_assocs.parquet')